In [ ]:
from common import *

## 12. Konačne predikcije

Zbog inherentne nebalansiranosti meteoroloških podataka (kišni dani čine manjinsku klasu), podrazumevani prag odlučivanja algoritma od $P \ge 0.50$ pokazao se kao previše restriktivan, što rezultira velikim brojem lažno negativnih predikcija (model propušta kišu). Aproksimacijom performansi kroz opseg pragova, u radu je primenjena optimizacija, čime je identifikovan novi prag koji pomera ravnotežu između preciznosti (Precision) i odziva (Recall), maksimizujući F1 ocenu.

Pored atributa same stanice, uvodimo i dva **prostorno-vremenska** atributa koja opisuju stanje obližnjih stanica *istog dana*: ponderisani prosek (inverzna udaljenost, isti princip kao u odeljku 11.2.2.2) procenta suseda sa `RainToday == 'Yes'` i ponderisani prosek `Pressure3pm` suseda, za sve stanice u okviru 300km (isti radijus na kom smo u odeljku 11.1.2 utvrdili da `Pressure` ostaje jako korelisan) - bez uključivanja same stanice. Ovo hvata signal da li sinoptički sistem (front, ciklon) već zahvata širi region, što je drugačija informacija od postojećih prostorno-statičnih obeležja (nadmorska visina, udaljenost od okeana), koja se ne menjaju iz dana u dan. Pošto koristimo isključivo podatke od *istog* dana sa drugih stanica (nikad iz budućnosti), ovo ne unosi curenje podataka - u produkciji bi ovi podaci bili dostupni u istom trenutku kada se predviđanje pravi.

**Zašto baš 300km, a ne manje?** Parovi stanica na manjoj razdaljini su međusobno jače korelisani po `RainToday` (prosečno 0.73 na <50km, 0.59 na 100-150km, 0.40 na 200-300km, ~0.01 preko 1000km), pa se moglo pomisliti da je 300km previše široko. Međutim, pošto se koristi ponderisani prosek (inverzna udaljenost), dalje stanice već dobijaju srazmerno manju težinu, pa smo direktno proverili ono što je bitno - korelaciju samog `Sused_RainToday_pct` sa `RainTomorrow`, i procenat redova bez ijednog suseda, za nekoliko radijusa:

- 100km: korelacija 0.289, nedostaje 46.8% redova
- 150km: korelacija 0.294, nedostaje 39.9% redova
- 200km: korelacija 0.284, nedostaje 28.1% redova
- 300km (izabrano): korelacija 0.282, nedostaje 14.7% redova
- 500km: korelacija 0.270, nedostaje 5.3% redova

Korelacija je praktično nepromenjena u celom ovom opsegu (0.27-0.29) - IDW ponderisanje efikasno potiskuje uticaj udaljenih stanica kad god postoji bliži sused, pa širenje radijusa ne razblažuje signal. Ono što se drastično menja jeste pokrivenost: mnoge stanice u australijskoj unutrašnjosti nemaju drugu stanicu bliže od 150-200km, pa manji radijus samo briše podatke bez ikakvog dobitka u kvalitetu. Zadržavamo 300km jer daje dobru pokrivenost (14.7% nedostaje, naspram npr. 39.9% na 150km) uz istu snagu signala, i poklapa se sa radijusom već uspostavljenim u odeljku 11.1.2 (gde je pritisak potvrđeno ostao jako korelisan i na 300km).

Ipak, u finalnom modelu (odeljak ispod) ova dva atributa NISU iskorišćena. Iz `feature_importances_` treniranog modela pokazalo se da su `Sused_RainToday_pct` i `Sused_Pressure3pm_avg` najmanje bitna dva od 12 atributa (svaki doprinosi svega ~2-3% odluke modela, naspram npr. ~31% za `Humidity3pm`). Kako bismo finalni model mogli jednostavno da serviramo (bez potrebe da se u produkciji istovremeno prikupljaju podaci sa svih susednih stanica za svaku predikciju), odlučili smo da ih izostavimo iz finalnog feature seta - istraživanje i korelaciona analiza ostaju iznad kao dokumentacija zašto smo ih uopšte razmatrali.

In [ ]:
def dodaj_susedne_atribute(df, putanja_do_udaljenosti="GeoPodaci/udaljenost_stanica.csv", max_dist=300.0):
    df_dist = pd.read_csv(putanja_do_udaljenosti)
    dist_dict = {}
    for _, red in df_dist.iterrows():
        s1, s2, d = red['stanica_1'], red['stanica_2'], red['razdaljina_km']
        if d <= max_dist:
            dist_dict.setdefault(s1, {})[s2] = d
            dist_dict.setdefault(s2, {})[s1] = d

    df = df.copy()
    # RainToday je po definiciji binarna verzija Rainfall-a (odeljak 6.4.2.6) - gde RainToday
    # nedostaje a Rainfall je poznat, mozemo ga determinsticki rekonstruisati istim pravilom
    # koje se vec koristi za proveru konzistentnosti (R3, odeljak 5.2), umesto da ga tretiramo
    # kao nepoznat i time nepotrebno smanjujemo pokrivenost novog atributa.
    raintoday_efektivno = df['RainToday'].where(
        df['RainToday'].notna() | df['Rainfall'].isna(),
        np.where(df['Rainfall'] >= 1.0, 'Yes', 'No')
    )
    df['_RainToday_bin'] = np.where(raintoday_efektivno == 'Yes', 1.0, np.where(raintoday_efektivno == 'No', 0.0, np.nan))

    pivot_kisa = df.pivot_table(index='Date', columns='Location', values='_RainToday_bin')
    pivot_pritisak = df.pivot_table(index='Date', columns='Location', values='Pressure3pm')

    lokacije = pivot_kisa.columns.tolist()
    tezine = pd.DataFrame(0.0, index=lokacije, columns=lokacije)
    for loc in lokacije:
        for sused, d in dist_dict.get(loc, {}).items():
            if sused in tezine.columns:
                tezine.loc[loc, sused] = 1.0 / (d + 1.0)

    def ponderisani_prosek_suseda(pivot):
        prisutno = pivot.notna().astype(float)
        vrednosti = pivot.fillna(0.0)
        brojioc = vrednosti.values @ tezine.values.T
        imenilac = prisutno.values @ tezine.values.T
        with np.errstate(invalid='ignore', divide='ignore'):
            rezultat = np.where(imenilac > 0, brojioc / imenilac, np.nan)
        return pd.DataFrame(rezultat, index=pivot.index, columns=pivot.columns)

    susedi_kisa = ponderisani_prosek_suseda(pivot_kisa).stack().rename('Sused_RainToday_pct').reset_index()
    susedi_pritisak = ponderisani_prosek_suseda(pivot_pritisak).stack().rename('Sused_Pressure3pm_avg').reset_index()

    df = df.merge(susedi_kisa, on=['Date', 'Location'], how='left')
    df = df.merge(susedi_pritisak, on=['Date', 'Location'], how='left')
    df.drop(columns=['_RainToday_bin'], inplace=True)
    return df

In [ ]:
def optimizuj_prag_odlucivanja(y_test, y_prob):
    
    pragovi = np.arange(0.1, 0.82, 0.02) 
    f1_skorovi = []

    for prag in pragovi:
        y_pred_custom = (y_prob >= prag).astype(int)
        f1 = f1_score(y_test, y_pred_custom)
        f1_skorovi.append(f1)

    najbolji_indeks = np.argmax(f1_skorovi)
    optimalni_prag = pragovi[najbolji_indeks]
    
    return optimalni_prag

In [ ]:

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000"))

mlflow.set_experiment("WeatherAus proba 1")

class RainPredictor:

    def __init__(self, filepath: str):
        self.filepath = filepath
        self.df = None

    def ucitaj_podatke(self):
        self.df = pd.read_csv(self.filepath)
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        self.df = self.df.sort_values(by='Date')
        self.df.dropna(subset=['RainTomorrow'], inplace=True)
        self.df.drop(columns=["Cloud9am", "RainToday", "Cloud3pm", "Evaporation", "Location", "TackaRose9am_missing", "TackaRose3pm_missing", "TackaRose9am", "TackaRose3pm"], inplace=True)
        if "Unnamed: 0.2" in self.df.columns:
            self.df.drop(columns=["Unnamed: 0.2"], inplace=True)
        if "Unnamed: 0" in self.df.columns:
            self.df.drop(columns=["Unnamed: 0"], inplace=True)
        if "Unnamed: 0.1" in self.df.columns:
            self.df.drop(columns=["Unnamed: 0.1"], inplace=True)
        
    def _pretprocesiraj_skup(self, data: pd.DataFrame) -> tuple:
        df_processed = data.copy()
        X = df_processed.drop(columns=["RainTomorrow"])
        y = df_processed['RainTomorrow'].map({'No': 0, 'Yes': 1})
        return X, y

    def _napravi_model(self, model_type: str, use_class_weight: bool):
        # Poredimo RandomForest (originalni model) sa XGBoost-om (pobedio na ROC-AUC/F1
        # u ablation testu, accuracy_push_ablation.py) u dva rezima:
        # (a) scale_pos_weight/class_weight + F1-optimalan prag - najbolji F1/recall
        #     (preporuceno kod neuravnotezenih klasa i asimetricne cene greske -
        #     propustanje stvarne kise je skuplje od laznog alarma);
        # (b) bez tezinjenja klasa + fiksan prag 0.50 - visi accuracy, nizi recall.
        if model_type == 'xgb':
            # Konfiguracija pronadjena 25-pokusajnom nasumicnom pretragom hiperparametara
            # (xgb_hyperparam_search.py), selektovanom na internoj validaciji izvucenoj
            # Iz train dela (hronoloski poslednjih 15% train perioda - test skup nikad
            # nije koriscen za izbor hiperparametara). scale_pos_weight=1.5 je pobedio
            # punu klasnu razmeru (~3.4). Ensembling (RF+XGB+LightGBM - ensemble_stacking.py)
            # je testiran i nije doneo merljivo poboljsanje, pa je zadrzan pojedinacan model.
            #
            # Napomena (overfitting_check.py): prvobitna konfiguracija (max_depth=8,
            # n_estimators=600) je imala veliki train-test gap (F1 gap 0.162, train F1=0.824
            # vs test F1=0.682 - klasican overfitting, model pamti trening deo bez ikakve
            # koristi na test delu). xgb_regularization_search.py je testirao 8 konzervativnijih
            # konfiguracija sa early stopping-om (na istoj internoj train-validaciji) i pronasao
            # konfiguraciju koja smanjuje gap za 63% (0.162 -> 0.060) uz prakticno identicnu test
            # performansu (F1 0.6820 vs 0.6822, ROC-AUC 0.8973 vs 0.8981) - znatno robusniji model.
            scale_pos_weight = 1.5 if use_class_weight else 1.0
            return XGBClassifier(
                n_estimators=1625, max_depth=4, learning_rate=0.03,
                subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
                gamma=0.2, reg_alpha=1.0, reg_lambda=2.0,
                random_state=42, n_jobs=-1, eval_metric='logloss',
                scale_pos_weight=scale_pos_weight,
            )
        elif model_type == 'rf':
            return RandomForestClassifier(
                n_estimators=100, random_state=42, n_jobs=-1,
                class_weight=('balanced' if use_class_weight else None),
                min_samples_leaf=5,
            )
        else:
            return DecisionTreeClassifier(
                random_state=42,
                class_weight=('balanced' if use_class_weight else None),
                min_samples_leaf=5,
                max_depth=10,
            )

    def treniraj_i_evaluiraj(self, ime_skupa: str, atributi, model_type: str = 'xgb', use_class_weight: bool = True, prag_fiksni: float = None):
        self.atributi = atributi
        with mlflow.start_run(run_name="unified_noleak_" + ime_skupa):
            print(f"\n{'='*50}\nProcesiranje i modelovanje za: {ime_skupa}\n{'='*50}")

            X_full, y = self._pretprocesiraj_skup(self.df)
            is_train = X_full['Date'] < SPLIT_DATE
            dates = X_full['Date']
            X = X_full[self.atributi]
            X_train, X_test = X[is_train], X[~is_train]
            y_train, y_test = y[is_train], y[~is_train]
            num_cols = X.select_dtypes(include=['float64', 'int64']).columns

            preprocessor = ColumnTransformer(
                transformers=[
                    ('num', StandardScaler(), num_cols)
                ]
            )
            pipeline = Pipeline(steps=[
                ('preprocesiranje', preprocessor),
                ('model', self._napravi_model(model_type, use_class_weight))
            ])
            pipeline.fit(X_train, y_train)
            y_prob = pipeline.predict_proba(X_test)[:, 1]

            if prag_fiksni is not None:
                optimalni_prag = prag_fiksni
                print(f"Koristi se fiksan prag: {optimalni_prag:.2f} (bez F1-optimizacije)")
            else:
                # Naslepo: prag se bira na odvojenoj internoj validaciji iz train dela
                # (hronoloski, poslednjih 15% train perioda) - Nikad na test skupu. Model
                # koji generise verovatnoce za izbor praga je treniran samo na preostalih
                # 85% train perioda (nikad video ni validaciju ni test), da izbor praga
                # bude potpuno nezavisan od test podataka.
                # (threshold_leakage_check.py: razlika u F1 izmedju "prevareno" i "naslepo"
                # pristupa je bila samo ~0.002 - mali uticaj, ali ovo je metodoloski cist pristup.)
                train_dates = dates[is_train]
                val_cutoff = train_dates.quantile(0.85)
                is_fit = is_train & (dates < val_cutoff)
                is_val = is_train & (dates >= val_cutoff)

                X_fit, y_fit = X[is_fit], y[is_fit]
                X_val, y_val = X[is_val], y[is_val]

                pipeline_fit = Pipeline(steps=[
                    ('preprocesiranje', ColumnTransformer(transformers=[('num', StandardScaler(), num_cols)])),
                    ('model', self._napravi_model(model_type, use_class_weight))
                ])
                pipeline_fit.fit(X_fit, y_fit)
                y_prob_val = pipeline_fit.predict_proba(X_val)[:, 1]
                optimalni_prag = optimizuj_prag_odlucivanja(y_val, y_prob_val)
                print(f"Prag odredjen naslepo na internoj validaciji ({len(X_val)} redova, "
                      f"nikad video test skup): {optimalni_prag:.2f}")
            mlflow.log_params({"optimalni_prag": optimalni_prag, "use_class_weight": use_class_weight})
            y_pred_optimalno = (y_prob >= optimalni_prag).astype(int)

            print(f"\nFinalni izveštaj klasifikacije sa pragom od {optimalni_prag:.2f}:")
            print(classification_report(y_test, y_pred_optimalno))

            acc = accuracy_score(y_test, y_pred_optimalno)
            roc_auc = roc_auc_score(y_test, y_prob)
            f1 = f1_score(y_test, y_pred_optimalno)
            precision = precision_score(y_test, y_pred_optimalno)
            recall = recall_score(y_test, y_pred_optimalno)
            mlflow.log_metric("acc", acc)
            mlflow.log_metric("roc_auc", roc_auc)
            mlflow.log_metric("f1", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            print(f"Tačnost (Accuracy): {acc:.4f}")
            print(f"ROC AUC Skor: {roc_auc:.4f}")
            print(f"F1 Skor (Klasa 1):  {f1:.4f}")
            print(f"Preciznost (Klasa 1): {precision:.4f}")
            print(f"Odziv (Klasa 1): {recall:.4f}")
            mlflow.sklearn.log_model(pipeline, name="moj_model " + ime_skupa, serialization_format="cloudpickle")


DATASET_PATH = "backups/WeatherAus_After_11_2_3_8.csv"

predictor = RainPredictor(filepath=DATASET_PATH)
predictor.ucitaj_podatke()

# Napomena: nakon popravke data leakage-a (train/test split odmah posle sistemskih/fizickih
# provera, pre detekcije anomalija i imputacije - videti sekciju "1.2b"), sve kolone
# (ukljucujuci Sunshine) su imputirane fit-ujuci modele samo na train delu. Isti SPLIT_DATE
# se koristi ovde za finalni klasifikator, umesto ranijeg
# train_test_split(shuffle=False, test_size=0.2).
ATRIBUTI_UNIFIED = ['Rainfall', 'Sunshine', 'WindGustSpeed', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Max_Min_Temp_Diff', 'Temp3pm_rolling_mean_3', 'Pressure3pm_pre_1_dana']

# (a) Decision Tree, class_weight=balanced, F1-optimalan prag - jednostavan baseline
predictor.treniraj_i_evaluiraj(ime_skupa='stablo_tezinsko_f1', atributi=ATRIBUTI_UNIFIED,
                                model_type='dt', use_class_weight=True, prag_fiksni=None)

# (b) RandomForest, class_weight=balanced, F1-optimalan prag - originalni model (poredjenje)
predictor.treniraj_i_evaluiraj(ime_skupa='suma_tezinsko_f1', atributi=ATRIBUTI_UNIFIED,
                                model_type='rf', use_class_weight=True, prag_fiksni=None)

# (b) XGBoost + scale_pos_weight + F1-optimalan prag - preporuceni finalni model
predictor.treniraj_i_evaluiraj(ime_skupa='xgboost_tezinsko_f1', atributi=ATRIBUTI_UNIFIED,
                                model_type='xgb', use_class_weight=True, prag_fiksni=None)

# (c) XGBoost bez tezinjenja klasa + fiksan prag 0.50 - visi accuracy, nizi recall (poredjenje)
predictor.treniraj_i_evaluiraj(ime_skupa='xgboost_osnovno_05', atributi=ATRIBUTI_UNIFIED,
                                model_type='xgb', use_class_weight=False, prag_fiksni=0.50)